# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string in Step 0, then run each cell in order. You will load RAG chunks, create vector and BM25 search indexes, retrieve context with vector and hybrid search, and assemble a grounded prompt for a chat model.

The final cell prints the prompt that your application would send to Azure OpenAI or Azure AI Foundry. The lab keeps the model call out of scope so the database retrieval mechanics are visible.


## Step 0: Connect to Azure DocumentDB

This cell loads the MongoDB Node.js driver, accepts your connection string, and opens the `docdbworkshop.rag_chunks` collection.

In [ ]:
let mongodb;
try { mongodb = require("mongodb"); } catch {
  const { execSync } = require("child_process");
  execSync("npm install mongodb", { stdio: "inherit" });
  mongodb = require("mongodb");
}
const { MongoClient } = mongodb;
const connectionString = process.env.DOCUMENTDB_CONNECTION_STRING || "<paste-your-azure-documentdb-connection-string-here>";
if (connectionString.includes("<paste")) throw new Error("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
const client = new MongoClient(connectionString);
await client.connect();
const db = client.db("docdbworkshop");
const chunks = db.collection("rag_chunks");
await db.command({ ping: 1 });

## Step 1: Load RAG chunks

Each chunk stores source metadata, text, tags, and an embedding together.

In [ ]:
const ragDocs = [
  { _id: "rag-001", sourceId: "search-module", title: "Vector search", chunk: "Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.", url: "module-4-search", tags: ["vector", "search"], embedding: [0.92, 0.80, 0.18] },
  { _id: "rag-002", sourceId: "search-module", title: "Full-text search", chunk: "Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.", url: "module-4-search", tags: ["full-text", "bm25"], embedding: [0.20, 0.12, 0.94] },
  { _id: "rag-003", sourceId: "search-module", title: "Hybrid search", chunk: "Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.", url: "module-4-search", tags: ["hybrid", "rrf"], embedding: [0.76, 0.70, 0.42] },
  { _id: "rag-004", sourceId: "rag-module", title: "Grounded generation", chunk: "A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.", url: "module-5-rag", tags: ["rag", "generation"], embedding: [0.84, 0.73, 0.34] }
];
await chunks.deleteMany({});
await chunks.insertMany(ragDocs);
await chunks.countDocuments({});

## Step 2: Create retrieval indexes

Create one vector index and one full-text search index on the same collection.

In [ ]:
await db.command({ createIndexes: "rag_chunks", indexes: [{ name: "idx_chunk_embedding_diskann", key: { embedding: "cosmosSearch" }, cosmosSearchOptions: { kind: "vector-diskann", dimensions: 3, similarity: "COS", maxDegree: 32, lBuild: 64 } }] });
await db.command({ createSearchIndexes: "rag_chunks", indexes: [{ name: "idx_chunk_fts", definition: { mappings: { dynamic: false, fields: { chunk: { type: "string" } } } } }] });

## Step 3: Retrieve context with vector search

The query vector represents the user's question and retrieves semantically similar chunks.

In [ ]:
const question = "How does DocumentDB retrieve context for RAG?";
const questionVector = [0.83, 0.74, 0.33];
const vectorContext = await chunks.aggregate([
  { $search: { cosmosSearch: { path: "embedding", vector: questionVector, k: 3 } } },
  { $project: { _id: 1, title: 1, chunk: 1, url: 1, score: { $meta: "searchScore" } } }
]).toArray();
vectorContext;

## Step 4: Retrieve context with hybrid search

Hybrid retrieval fuses BM25 and vector results with RRF, which works well for RAG questions that mix natural language and exact terms.

In [ ]:
const keywordContext = await chunks.aggregate([
  { $search: { index: "idx_chunk_fts", text: { query: question, path: "chunk" } } },
  { $limit: 3 },
  { $project: { _id: 1, title: 1, chunk: 1, url: 1, score: { $meta: "searchScore" } } }
]).toArray();
function rrf(lists, k = 60, topN = 3) {
  const scores = new Map(); const docsById = new Map();
  for (const list of lists) {
    list.forEach((doc, rank) => {
      const id = doc._id.toString(); docsById.set(id, doc);
      scores.set(id, (scores.get(id) ?? 0) + 1 / (k + rank + 1));
    });
  }
  return [...scores.entries()].sort((a, b) => b[1] - a[1]).slice(0, topN).map(([id, score]) => ({ ...docsById.get(id), rrfScore: score }));
}
const hybridContext = rrf([keywordContext, vectorContext]);
hybridContext;

## Step 5: Build the grounded prompt

This is the prompt your application sends to the chat model after retrieval.

In [ ]:
const contextBlock = hybridContext.map((doc, i) => `[${i + 1}] ${doc.title}
${doc.chunk}
Source: ${doc.url}`).join("

");
const groundedPrompt = `You are a helpful assistant for an Azure DocumentDB workshop.
Answer the user's question using only the context below.
If the context does not contain the answer, say you do not know based on the provided context.

<context>
${contextBlock}
</context>

Question: ${question}`;
console.log(groundedPrompt);